<a href="https://colab.research.google.com/github/JeysonCarmona/Titanic-Practice/blob/main/Titanic_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Jeyson Carmona Bedoya

#  Titanic — Survival Prediction
## Kaggle Competition | Binary Classification

The sinking of the Titanic in April 1912 is one of the most infamous maritime disasters in history. Out of 2,224 passengers and crew, approximately 1,502 died — many due to a shortage of lifeboats. Survival was not random: women, children, and first-class passengers were significantly more likely to survive.

**Objective:** Build a machine learning model that predicts which passengers survived the Titanic shipwreck, using features such as passenger class, sex, age, and fare.

**Dataset:** The data is split into a training set (`train.csv`, 891 rows) and a test set (`test.csv`, 418 rows) provided by Kaggle. The target variable is `Survived` (0 = No, 1 = Yes).

**Approach:**
1. Exploratory Data Analysis (EDA) — understand distributions, correlations, and survival patterns
2. Feature Engineering — extract meaningful signals from raw features
3. Preprocessing & Encoding — prepare data for ML models
4. Model Training & Hyperparameter Tuning — compare multiple classifiers
5. Ensembling — Voting and Stacking to improve generalization


## 1. Environment Setup

We begin by importing the core libraries for data manipulation (`numpy`, `pandas`), visualization (`matplotlib`, `seaborn`, `plotly`), and machine learning (`scikit-learn`). Warnings are suppressed to keep the output clean during development.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set()

# plotly library
import plotly.graph_objs as go
from plotly import tools
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True)


import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
#warnings.filterwarnings("ignore")
#warnings.filterwarnings(module='sklearn*', action='ignore', category=DeprecationWarning)
#warnings.filterwarnings(action='once')

#from sklearn.utils.testing import ignore_warnings # Removed this line

from subprocess import check_output
print(check_output(["ls", "/content"]).decode("utf8")) # Changed ../input to /content

sample_data



In [2]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

### Utility Functions

Two helper functions are defined upfront to avoid code repetition throughout the notebook:
- `get_best_score`: prints and returns the best cross-validation score, parameters, and estimator after a `GridSearchCV`.
- `plot_feature_importances`: visualizes the top 10 most important features for tree-based models.


In [3]:
def get_best_score(model):

    print(model.best_score_)
    print(model.best_params_)
    print(model.best_estimator_)

    return model.best_score_


def plot_feature_importances(model, columns):
    nr_f = 10
    imp = pd.Series(data = model.best_estimator_.feature_importances_,
                    index=columns).sort_values(ascending=False)
    plt.figure(figsize=(7,5))
    plt.title("Feature importance")
    ax = sns.barplot(y=imp.index[:nr_f], x=imp.values[:nr_f], orient='h')

### Loading the Data

We load both the training and test sets. It is important to keep them separate throughout all preprocessing steps — information from the test set must never influence any transformation fitted on the training data, to avoid **data leakage**.


In [8]:
# ── Carga de datos desde GitHub ──────────────────────────────────────────────
GITHUB_USER = "JeysonCarmona"
REPO        = "Titanic-Practice"
BRANCH      = "main"

base_url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/{BRANCH}/data"

df_train = pd.read_csv(f"{base_url}/train.csv")
df_test  = pd.read_csv(f"{base_url}/test.csv")

print(f"Train: {df_train.shape} | Test: {df_test.shape}")

Train: (891, 12) | Test: (418, 11)


---
## 2. Exploratory Data Analysis (EDA)

Before building any model, we need to deeply understand the data: its structure, distributions, missing values, and how each feature relates to the survival outcome. A thorough EDA prevents us from making uninformed modelling decisions later.


### 2.1 Initial Data Inspection

We start by previewing the first rows of both datasets and checking dtypes and non-null counts. This reveals the feature space and highlights columns that may require attention.

| Feature | Type | Description |
|---|---|---|
| `Pclass` | Categorical (ordinal) | Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd) |
| `Sex` | Categorical | Passenger gender |
| `Age` | Continuous | Age in years (19% missing in train) |
| `SibSp` | Discrete | # of siblings/spouses aboard |
| `Parch` | Discrete | # of parents/children aboard |
| `Fare` | Continuous | Passenger fare |
| `Embarked` | Categorical | Port of embarkation (C, Q, S) |
| `Cabin` | String | Cabin number (~77% missing — will be dropped) |


In [ ]:
df_train.head()

In [ ]:
df_test.head()


In [ ]:
df_train.info()


In [ ]:
df_test.info()


### 2.1.1 Descriptive Statistics & Numerical Correlations

Beyond inspecting dtypes, we compute descriptive statistics and the **point-biserial correlation** between each numerical feature and the target variable `Survived`. This gives us a quick quantitative ranking of feature relevance before any visualisation.

> A positive correlation means higher values of that feature are associated with higher survival; negative means the opposite.


In [ ]:
# --- Descriptive statistics for numerical columns ---
print("=== Training Set — Descriptive Statistics ===")
display(df_train.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std']))

# --- Point-biserial correlation with Survived ---
num_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
correlations = df_train[num_cols + ['Survived']].corr()['Survived'].drop('Survived').sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#d73027' if v < 0 else '#1a9850' for v in correlations.values]
bars = ax.barh(correlations.index, correlations.values, color=colors, edgecolor='white', height=0.6)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Pearson Correlation with Survived', fontsize=12)
ax.set_title('Numerical Feature Correlation with Survival', fontsize=14, fontweight='bold', pad=12)
for bar, val in zip(bars, correlations.values):
    ax.text(val + (0.005 if val >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10)
ax.set_xlim(-0.4, 0.35)
sns.despine()
plt.tight_layout()
plt.show()

print("\nCorrelation values:")
print(correlations.to_string())


### 2.1.2 Statistical Significance — Chi-Squared Tests (Categorical Features)

For categorical features, Pearson's chi-squared test of independence checks whether there is a **statistically significant association** with `Survived`. The null hypothesis is that the feature and survival are independent.

- A **p-value < 0.05** means we reject the null hypothesis — the feature has a significant relationship with survival.
- **Cramér's V** measures the *strength* of that association (0 = no association, 1 = perfect association), making it comparable across features with different numbers of categories.


In [ ]:
from scipy.stats import chi2_contingency
import numpy as np

cat_cols = ['Sex', 'Pclass', 'Embarked', 'SibSp', 'Parch']
results = []

for col in cat_cols:
    ct = pd.crosstab(df_train[col], df_train['Survived'])
    chi2, p, dof, _ = chi2_contingency(ct)
    n = ct.values.sum()
    cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    results.append({
        'Feature': col,
        'Chi2 Statistic': round(chi2, 2),
        'p-value': f'{p:.2e}',
        'Degrees of Freedom': dof,
        "Cramér's V": round(cramers_v, 4),
        'Significant (p<0.05)': ' Yes' if p < 0.05 else ' No'
    })

df_chi2 = pd.DataFrame(results).sort_values("Cramér's V", ascending=False)
display(df_chi2.set_index('Feature').style.background_gradient(
    cmap='YlGn', subset=["Cramér's V"]))

# --- Bar chart of Cramér's V ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(df_chi2['Feature'], df_chi2["Cramér's V"],
        color='#2c7bb6', edgecolor='white', height=0.6)
ax.axvline(0.1, color='orange', linestyle='--', linewidth=1.2, label='Weak (0.1)')
ax.axvline(0.3, color='red',    linestyle='--', linewidth=1.2, label='Moderate (0.3)')
ax.set_xlabel("Cramér's V (Association Strength)", fontsize=12)
ax.set_title("Categorical Feature Association with Survival\n(Chi-Squared Test)",
             fontsize=13, fontweight='bold', pad=10)
ax.legend(fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()


### 2.1.3 Survival Rate Summary by Key Groups

A structured summary table gives a quick quantitative snapshot of how survival rates differ across the most important groupings. This complements the visual analysis and provides concrete numbers to reference.


In [ ]:
# Survival rate summary by key groupings
summary_data = []

for col in ['Sex', 'Pclass', 'Embarked']:
    grp = df_train.groupby(col)['Survived'].agg(['mean', 'count', 'sum'])
    grp.columns = ['Survival Rate', 'Total Passengers', 'Survivors']
    grp['Survival Rate'] = (grp['Survival Rate'] * 100).round(1).astype(str) + '%'
    grp.index.name = col
    grp['Group'] = col
    grp = grp.reset_index().rename(columns={col: 'Value'})
    summary_data.append(grp)

df_summary = pd.concat(summary_data, ignore_index=True)[
    ['Group', 'Value', 'Total Passengers', 'Survivors', 'Survival Rate']]

display(df_summary.set_index(['Group', 'Value']).style
    .set_caption("Survival Rate by Key Categorical Groups")
    .background_gradient(cmap='RdYlGn', subset=['Survivors'])
)

# Age group survival
print("\n--- Survival by Age Group (deciles) ---")
df_train['AgeGroup'] = pd.cut(df_train['Age'], bins=[0,12,18,35,60,100],
                               labels=['Child','Teen','Adult','Middle-aged','Senior'])
age_surv = df_train.groupby('AgeGroup', observed=True)['Survived'].agg(['mean','count'])
age_surv.columns = ['Survival Rate', 'Count']
age_surv['Survival Rate'] = (age_surv['Survival Rate'] * 100).round(1).astype(str) + '%'
display(age_surv)
df_train.drop('AgeGroup', axis=1, inplace=True)  # clean up temp column


### 2.2 Missing Value Analysis

Understanding the extent and pattern of missing data is critical before any imputation. We visualize missingness with a heatmap — light colors indicate missing cells.

**Key findings:**
- `Age`: ~19% missing in train, ~20% in test — requires careful imputation
- `Cabin`: ~77% missing — too sparse to be directly useful; will be dropped
- `Embarked`: 2 missing values in train — trivial to impute with the mode
- `Fare`: 1 missing value in test — will be filled with the mean


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
sns.heatmap(df_train.isnull(), cbar=False, cmap="YlGnBu_r")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
sns.heatmap(df_test.isnull(), cbar=False, cmap="YlGnBu_r")
plt.show()


### 2.3 Categorical Feature Analysis

We explore the relationship between each categorical feature and the survival rate. The countplots below show the distribution of survivors vs. non-survivors across `Sex`, `Pclass`, `SibSp`, `Parch`, and `Embarked`.


In [ ]:
cols = ['Survived', 'Sex', 'Pclass', 'SibSp', 'Parch', 'Embarked']

In [ ]:
nr_rows = 2
nr_cols = 3

fig, axs = plt.subplots(nr_rows, nr_cols, figsize=(nr_cols*3.5,nr_rows*3))

for r in range(0,nr_rows):
    for c in range(0,nr_cols):

        i = r*nr_cols+c
        ax = axs[r][c]
        sns.countplot(x=cols[i], hue="Survived", data=df_train, ax=ax)
        ax.set_title(cols[i], fontsize=14, fontweight='bold')
        ax.legend(title="survived", loc='upper center')

plt.tight_layout()

### 2.4 Age Distribution by Sex, Class, and Survival

The distribution of `Age` varies substantially across passenger classes and sex. We use a `FacetGrid` to decompose these three variables simultaneously, revealing that young children (especially in 3rd class) had mixed survival rates, while women across all classes had notably higher survival.


In [ ]:
bins = np.arange(0, 80, 5)
g = sns.FacetGrid(df_train, row='Sex', col='Pclass', hue='Survived', margin_titles=True, height=3, aspect=1.1)
g.map(sns.distplot, 'Age', kde=False, bins=bins, hist_kws=dict(alpha=0.6))
g.add_legend()
plt.show()

In [ ]:
df_train['Fare'].max()

### 2.5 Fare Distribution by Sex, Class, and Survival

Higher fares are strongly associated with 1st class tickets and higher survival probability. The right-skewed distribution of `Fare` suggests that binning it into intervals will be more informative for models than the raw continuous value.


In [ ]:
bins = np.arange(0, 550, 50)
g = sns.FacetGrid(df_train, row='Sex', col='Pclass', hue='Survived', margin_titles=True, height=3, aspect=1.1)
g.map(sns.distplot, 'Fare', kde=False, bins=bins, hist_kws=dict(alpha=0.6))
g.add_legend()
plt.show()

### 2.6 Survival Rates by Key Variables

We compute and visualize the mean survival rate across `Pclass`, `Sex`, and `Embarked`. These are among the most predictive features in the dataset.

> **Insight:** Female passengers had a survival rate of ~74%, vs. ~19% for males. First-class passengers survived at ~63%, while third-class at ~24%. The "women and children first" protocol is clearly reflected in the data.


In [ ]:
sns.barplot(x='Pclass', y='Survived', data=df_train)
plt.ylabel("Survival Rate")
plt.title("Survival as function of Pclass")
plt.show()

In [ ]:
sns.barplot(x='Sex', y='Survived', hue='Pclass', data=df_train)
plt.ylabel("Survival Rate")
plt.title("Survival as function of Pclass and Sex")
plt.show()

### 2.7 Embarkation Port Analysis

Passengers who embarked at Cherbourg (`C`) had a significantly higher survival rate. However, this is largely explained by the class composition: Cherbourg had a higher proportion of 1st-class passengers. The raw `Embarked` feature carries indirect socioeconomic signal.


In [ ]:
sns.barplot(x='Embarked', y='Survived', data=df_train)
plt.ylabel("Survival Rate")
plt.title("Survival as function of Embarked Port")
plt.show()

In [ ]:
sns.barplot(x='Embarked', y='Survived', hue='Pclass', data=df_train)
plt.ylabel("Survival Rate")
plt.title("Survival as function of Embarked Port")
plt.show()

In [ ]:
sns.countplot(x='Embarked', hue='Pclass', data=df_train)
plt.title("Count of Passengers as function of Embarked Port")
plt.show()

In [ ]:
sns.boxplot(x='Embarked', y='Age', data=df_train)
plt.title("Age distribution as function of Embarked Port")
plt.show()

In [ ]:
sns.boxplot(x='Embarked', y='Fare', data=df_train)
plt.title("Fare distribution as function of Embarked Port")
plt.show()

In [ ]:
cm_surv = ["darkgrey" , "lightgreen"]

### 2.8 Multivariable Analysis: Age × Pclass × Survival

We use swarm and violin plots to visualize the joint distribution of `Age` and `Pclass`, colored by survival. These plots reveal nuanced patterns that bar charts cannot capture — for example, that young male passengers in 3rd class had dramatically lower survival odds, while elderly passengers in 1st class show higher variance.


In [ ]:
fig, ax = plt.subplots(figsize=(13,7))
sns.swarmplot(x='Pclass', y='Age', hue='Survived', data=df_train , palette=cm_surv, size=7, ax=ax)
plt.title('Survivals for Age and Pclass ')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13,7))
sns.violinplot(x="Pclass", y="Age", hue='Survived', data=df_train, split=True, bw=0.05 , palette=cm_surv, ax=ax)
plt.title('Survivals for Age and Pclass ')
plt.show()

In [ ]:
g = sns.catplot(x="Pclass", y="Age", hue="Survived", col="Sex", data=df_train, kind="swarm", palette=cm_surv, height=7, aspect=.9, s=7)
plt.show()

In [ ]:
g = sns.catplot(x="Pclass", y="Age", hue="Survived", col="Sex", data=df_train, kind="violin", bw=0.05, palette=cm_surv, height=7, aspect=.9)
plt.show()

---
## 3. Feature Engineering

Raw features rarely capture the full predictive signal in the data. This section transforms and creates new variables to improve model performance.

We apply all transformations **simultaneously** to both `df_train` and `df_test` to ensure full consistency between the two datasets.


### 3.1 Family Size, Isolation Flag, and Name-Based Features

**`FamilySize`** = `SibSp` + `Parch` + 1 (the passenger themselves). Research shows that very large families and solo travelers had lower survival — medium-sized families (2–4 members) were best positioned to help each other.

**`Alone`** is a binary flag for passengers travelling without any family member.

**`NameLen`** captures the length of each passenger's name. While seemingly arbitrary, longer names tend to correlate with nobility or higher social status, which in turn correlates with class and survival. We bin it into intervals of 5 characters (`NameLenBin`).

**`Title`** is extracted from the name using a regex pattern. Titles encode both sex and social status (e.g., `Master` = young male, `Lady` / `Countess` = aristocracy). Rare titles are mapped to broader categories to reduce dimensionality.


In [ ]:
for df in [df_train, df_test] :

    df['FamilySize'] = df['SibSp'] + df['Parch'] +1

    df['Alone']=0
    df.loc[(df.FamilySize==1),'Alone'] = 1

    df['NameLen'] = df.Name.apply(lambda x : len(x))
    df['NameLenBin']=np.nan
    for i in range(20,0,-1):
        df.loc[ df['NameLen'] <= i*5, 'NameLenBin'] = i


    df['Title']=0
    df['Title']=df.Name.str.extract(r'([A-Za-z]+)\.') #lets extract the Salutations
    df['Title'].replace(['Mlle','Mme','Ms','Dr','Major','Lady','Countess','Jonkheer','Col','Rev','Capt','Sir','Don'],
                    ['Miss','Miss','Miss','Mr','Mr','Mrs','Mrs','Other','Other','Other','Mr','Mr','Mr'],inplace=True)

In [ ]:
print(df_train[['NameLen' , 'NameLenBin']].head(10))

In [ ]:
grps_namelenbin_survrate = df_train.groupby(['NameLenBin'])['Survived'].mean().to_frame()
grps_namelenbin_survrate

In [ ]:
plt.subplots(figsize=(10,6))
sns.barplot(x='NameLenBin' , y='Survived' , data = df_train)
plt.ylabel("Survival Rate")
plt.title("Survival as function of NameLenBin")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9,7))
sns.violinplot(x="NameLenBin", y="Pclass", data=df_train, hue='Survived', split=True,
               orient="h", bw=0.2 , palette=cm_surv, ax=ax)
plt.show()

In [ ]:
g = sns.catplot(x="NameLenBin", y="Survived", col="Sex", data=df_train, kind="bar", height=5, aspect=1.2)
plt.show()

In [ ]:
grps_title_survrate = df_train.groupby(['Title'])['Survived'].mean().to_frame()
grps_title_survrate

In [ ]:
plt.subplots(figsize=(10,6))
sns.barplot(x='Title' , y='Survived' , data = df_train)
plt.ylabel("Survival Rate")
plt.title("Survival as function of Title")
plt.show()

In [ ]:
pd.crosstab(df_train.FamilySize,df_train.Survived).apply(lambda r: r/r.sum(), axis=1).style.background_gradient(cmap='summer_r')

In [ ]:
plt.subplots(figsize=(10,6))
sns.barplot(x='FamilySize' , y='Survived' , data = df_train)
plt.ylabel("Survival Rate")
plt.title("Survival as function of FamilySize")
plt.show()

### 3.2 Missing Value Imputation

**Age imputation by Title group:** Instead of filling missing `Age` values with the global mean (which ignores important sub-population differences), we impute using the **mean age per title group**. A `Master` is typically a young boy, while a `Mr` is an adult male — filling them with the same value would introduce systematic bias.

**Embarked:** Only 2 missing values — imputed with the most frequent port (`S`).

**Fare:** Only 1 missing value in the test set — imputed with the mean fare.


In [ ]:
for df in [df_train, df_test]:

    # Title
    df['Title'] = df['Title'].fillna(df['Title'].mode().iloc[0])

    # Age: use Title to fill missing values
    df.loc[(df.Age.isnull())&(df.Title=='Mr'),'Age']= df.Age[df.Title=="Mr"].mean()
    df.loc[(df.Age.isnull())&(df.Title=='Mrs'),'Age']= df.Age[df.Title=="Mrs"].mean()
    df.loc[(df.Age.isnull())&(df.Title=='Master'),'Age']= df.Age[df.Title=="Master"].mean()
    df.loc[(df.Age.isnull())&(df.Title=='Miss'),'Age']= df.Age[df.Title=="Miss"].mean()
    df.loc[(df.Age.isnull())&(df.Title=='Other'),'Age']= df.Age[df.Title=="Other"].mean()
    df = df.drop('Name', axis=1)

In [ ]:
# Embarked
df_train['Embarked'] = df_train['Embarked'].fillna(df_train['Embarked'].mode().iloc[0])
df_test['Embarked'] = df_test['Embarked'].fillna(df_test['Embarked'].mode().iloc[0])

# Fare
df_train['Fare'] = df_train['Fare'].fillna(df_train['Fare'].mean())
df_test['Fare'] = df_test['Fare'].fillna(df_test['Fare'].mean())


### 3.3 Binning, Encoding, and Final Feature Set

**Age and Fare binning:** Continuous variables are discretized into bins (`Age_bin`, `Fare_bin`). This reduces the influence of outliers and helps tree-based models find more stable splits.

**Title encoding:** Titles are mapped to numeric codes before one-hot encoding is applied to the nominal categoricals (`Sex`, `Embarked`, `Pclass`).

**Dropped features:**
- `Name`, `Ticket`, `Cabin`: too sparse or no direct predictive signal after feature extraction
- `SibSp`, `Parch`, `Alone`: absorbed into `FamilySize`
- `NameLen`: absorbed into `NameLenBin`


In [ ]:
for df in [df_train, df_test]:

    df['Age_bin']=np.nan
    for i in range(8,0,-1):
        df.loc[ df['Age'] <= i*10, 'Age_bin'] = i

    df['Fare_bin']=np.nan
    for i in range(12,0,-1):
        df.loc[ df['Fare'] <= i*50, 'Fare_bin'] = i

    # convert Title to numerical
    title_mapping = {'Other':0, 'Mr': 1, 'Master':2, 'Miss': 3, 'Mrs': 4 }
    df['Title'] = df['Title'].map(title_mapping)

    # fill na with maximum frequency mode, or a default if mode is empty
    current_title_mode = df['Title'].mode()
    if current_title_mode.empty:
        # If all titles are NaN after mapping, fill with 0 ('Other')
        df['Title'] = df['Title'].fillna(0)
    else:
        # Otherwise, fill with the mode of the numerical titles
        df['Title'] = df['Title'].fillna(current_title_mode.iloc[0])

    df['Title'] = df['Title'].astype(int)

In [ ]:
df_train_ml = df_train.copy()
df_test_ml = df_test.copy()

passenger_id = df_test_ml['PassengerId']

In [ ]:
df_train_ml.info()


In [ ]:
df_test_ml.info()


In [ ]:

df_train_ml = pd.get_dummies(df_train_ml, columns=['Sex', 'Embarked', 'Pclass'], drop_first=True)
df_test_ml = pd.get_dummies(df_test_ml, columns=['Sex', 'Embarked', 'Pclass'], drop_first=True)

df_train_ml.drop(['PassengerId','Name','Ticket', 'Cabin', 'Age', 'Fare_bin'],axis=1,inplace=True)
df_test_ml.drop(['PassengerId','Name','Ticket', 'Cabin', 'Age', 'Fare_bin'],axis=1,inplace=True)

#df_train_ml.drop(['PassengerId','Name','Ticket', 'Cabin', 'Age_bin', 'Fare_bin'],axis=1,inplace=True)
#df_test_ml.drop(['PassengerId','Name','Ticket', 'Cabin', 'Age_bin', 'Fare_bin'],axis=1,inplace=True)

In [ ]:
df_train_ml.dropna(inplace=True)


In [ ]:
for df in [df_train_ml, df_test_ml]:
    df.drop(['NameLen'], axis=1, inplace=True)

    df.drop(['SibSp'], axis=1, inplace=True)
    df.drop(['Parch'], axis=1, inplace=True)
    df.drop(['Alone'], axis=1, inplace=True)

In [ ]:
df_train_ml.head()


In [ ]:
df_test_ml.fillna(df_test_ml.mean(), inplace=True)
df_test_ml.head()

### 3.4 Feature Scaling

Distance-based models (SVC, KNN) and gradient-based methods are sensitive to feature scale. We apply `StandardScaler` to produce zero-mean, unit-variance features.

> **Important:** The scaler is **fit exclusively on the training set** and then applied (`.transform()`) to both train and test. Fitting on the test set would constitute data leakage and produce overly optimistic estimates.

We maintain two versions of the data:
- `X` / `X_test`: unscaled (for tree-based models that are scale-invariant)
- `X_sc` / `X_test_sc`: scaled (for SVC, KNN, and boosting models)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# for df_train_ml
scaler.fit(df_train_ml.drop(['Survived'],axis=1))
scaled_features = scaler.transform(df_train_ml.drop(['Survived'],axis=1))
df_train_ml_sc = pd.DataFrame(scaled_features) # columns=df_train_ml.columns[1::])

# for df_test_ml
df_test_ml.fillna(df_test_ml.mean(), inplace=True)
#scaler.fit(df_test_ml)
scaled_features = scaler.transform(df_test_ml)
df_test_ml_sc = pd.DataFrame(scaled_features) # , columns=df_test_ml.columns)

In [ ]:
df_train_ml_sc.head()


In [ ]:
df_test_ml_sc.head()


In [ ]:
df_train_ml.head()


In [ ]:
X = df_train_ml.drop('Survived', axis=1)
y = df_train_ml['Survived']
X_test = df_test_ml

X_sc = df_train_ml_sc
y_sc = df_train_ml['Survived']
X_test_sc = df_test_ml_sc

---
## 4. Model Training, Hyperparameter Optimization, and Ensembling

We train and tune **11 classifiers**, covering a broad spectrum of ML families:

| Family | Models |
|---|---|
| Instance-based | K-Nearest Neighbors (KNN) |
| Tree-based | Decision Tree, Random Forest, Extra Trees |
| Kernel-based | Support Vector Classifier (SVC) |
| Gradient Boosting | GBC, XGBoost, AdaBoost, CatBoost, LightGBM |
| Ensemble meta-models | Voting Classifier, Stacking Classifier |

For all models, we use **10-fold stratified cross-validation** to estimate generalization performance, and either `GridSearchCV` or `RandomizedSearchCV` for hyperparameter tuning.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn import tree


from sklearn.metrics import accuracy_score

In [ ]:
from sklearn.model_selection import cross_val_score


### 4.1 Baseline — Support Vector Classifier (SVC)

We establish a baseline with manually chosen hyperparameters before tuning, to quantify the improvement gained by optimization.


In [ ]:
svc = SVC(gamma = 0.01, C = 100)
scores_svc = cross_val_score(svc, X, y, cv=10, scoring='accuracy')
print(scores_svc)
print(scores_svc.mean())

In [ ]:
svc = SVC(gamma = 0.01, C = 100)
scores_svc_sc = cross_val_score(svc, X_sc, y_sc, cv=10, scoring='accuracy')
print(scores_svc_sc)
print(scores_svc_sc.mean())

In [ ]:
rfc = RandomForestClassifier(max_depth=5, max_features=6)
scores_rfc = cross_val_score(rfc, X, y, cv=10, scoring='accuracy')
print(scores_rfc)
print(scores_rfc.mean())

### 4.1.1 Learning Curves — Diagnosing Bias vs. Variance

Before committing to full hyperparameter tuning, learning curves let us diagnose whether a model suffers from **high bias** (underfitting) or **high variance** (overfitting):

- If both training and validation scores are low → **underfitting**: the model needs more complexity or better features.
- If training score is high but validation score is low → **overfitting**: the model needs regularization or more data.
- If both scores converge and are high → **well-fitted model**.

We plot learning curves for the three most representative model families: SVC, Random Forest, and XGBoost.


In [ ]:
from sklearn.model_selection import learning_curve

def plot_learning_curve(estimator, X, y, title, cv=5, n_jobs=-1,
                        train_sizes=np.linspace(0.1, 1.0, 8)):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=n_jobs,
        train_sizes=train_sizes, scoring='accuracy')

    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    plt.plot(train_sizes, train_mean, 'o-', color='#2196F3', label='Training score')
    plt.plot(train_sizes, val_mean,   'o-', color='#FF5722', label='Cross-val score')
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#2196F3')
    plt.fill_between(train_sizes, val_mean - val_std,   val_mean + val_std,   alpha=0.15, color='#FF5722')
    plt.axhline(y=val_mean[-1], linestyle='--', color='grey', linewidth=0.9)
    plt.title(title, fontsize=13, fontweight='bold')
    plt.xlabel('Training Set Size')
    plt.ylabel('Accuracy')
    plt.legend(loc='lower right')
    plt.ylim(0.6, 1.02)
    sns.despine()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plt.sca(axes[0])
plot_learning_curve(SVC(C=100, gamma=0.01), X_sc, y_sc, 'SVC (C=100, γ=0.01)')

plt.sca(axes[1])
plot_learning_curve(RandomForestClassifier(max_depth=5, max_features=6, random_state=42),
                    X, y, 'Random Forest (depth=5)')

from xgboost import XGBClassifier
plt.sca(axes[2])
plot_learning_curve(XGBClassifier(max_depth=6, gamma=2, learning_rate=0.2, eval_metric='logloss', random_state=42),
                    X_sc, y_sc, 'XGBoost (depth=6, γ=2)')

plt.suptitle('Learning Curves — Bias-Variance Diagnosis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from scipy.stats import uniform

#### Hyperparameter Tuning — SVC

We use `RandomizedSearchCV` first to explore a broad continuous space for `C` and `gamma`, then refine with `GridSearchCV` over a discrete grid. The `rbf` kernel is preferred as it handles non-linear decision boundaries common in tabular data.


In [ ]:
model = SVC()
param_grid = {'C':uniform(0.1, 5000), 'gamma':uniform(0.0001, 1) }
rand_SVC = RandomizedSearchCV(model, param_distributions=param_grid, n_iter=100)
rand_SVC.fit(X_sc,y_sc)
score_rand_SVC = get_best_score(rand_SVC)

In [ ]:
param_grid = {'C': [0.1,10, 100, 1000,5000], 'gamma': [1,0.1,0.01,0.001,0.0001], 'kernel': ['rbf']}
svc_grid = GridSearchCV(SVC(), param_grid, cv=10, refit=True, verbose=1)
svc_grid.fit(X_sc,y_sc)
sc_svc = get_best_score(svc_grid)

In [ ]:
pred_all_svc = svc_grid.predict(X_test_sc)

sub_svc = pd.DataFrame()
sub_svc['PassengerId'] = df_test['PassengerId']
sub_svc['Survived'] = pred_all_svc
sub_svc.to_csv('svc.csv',index=False)

### 4.2 K-Nearest Neighbors (KNN)

KNN is a non-parametric model that classifies based on the majority vote of the `k` nearest neighbors. It requires scaled features. We tune `n_neighbors`, `leaf_size`, and the distance weighting strategy.


In [ ]:
knn = KNeighborsClassifier()
leaf_range = list(range(3, 15, 1))
k_range = list(range(1, 15, 1))
weight_options = ['uniform', 'distance']
param_grid = dict(leaf_size=leaf_range, n_neighbors=k_range, weights=weight_options)
print(param_grid)

knn_grid = GridSearchCV(knn, param_grid, cv=10, verbose=1, scoring='accuracy')
knn_grid.fit(X, y)

sc_knn = get_best_score(knn_grid)

In [ ]:
pred_all_knn = knn_grid.predict(X_test)

sub_knn = pd.DataFrame()
sub_knn['PassengerId'] = df_test['PassengerId']
sub_knn['Survived'] = pred_all_knn
sub_knn.to_csv('knn.csv',index=False)

### 4.3 Decision Tree

A single decision tree serves as an interpretable baseline for tree-based methods. We tune `min_samples_split` to control overfitting. Decision trees tend to overfit on this dataset, but they provide a useful lower bound for ensemble methods.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
dtree = DecisionTreeClassifier()

param_grid = {'min_samples_split': [4,7,10,12]}
dtree_grid = GridSearchCV(dtree, param_grid, cv=10, refit=True, verbose=1)
dtree_grid.fit(X_sc,y_sc)

print(dtree_grid.best_score_)
print(dtree_grid.best_params_)
print(dtree_grid.best_estimator_)

In [ ]:
pred_all_dtree = dtree_grid.predict(X_test_sc)

sub_dtree = pd.DataFrame()
sub_dtree['PassengerId'] = df_test['PassengerId']
sub_dtree['Survived'] = pred_all_dtree
sub_dtree.to_csv('dtree.csv',index=False)

### 4.4 Random Forest

Random Forest improves upon a single decision tree by averaging the predictions of many decorrelated trees, each trained on a random bootstrap sample and a random subset of features. This significantly reduces variance without increasing bias. We tune `max_depth`, `max_features`, and `min_samples_split`.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
rfc = RandomForestClassifier()

param_grid = {'max_depth': [3, 5, 6, 7, 8], 'max_features': [6,7,8,9,10],
              'min_samples_split': [5, 6, 7, 8]}

rf_grid = GridSearchCV(rfc, param_grid, cv=10, refit=True, verbose=1)
rf_grid.fit(X_sc,y_sc)
sc_rf = get_best_score(rf_grid)

In [ ]:
plot_feature_importances(rf_grid, X.columns)

In [ ]:
pred_all_rf = rf_grid.predict(X_test_sc)

sub_rf = pd.DataFrame()
sub_rf['PassengerId'] = df_test['PassengerId']
sub_rf['Survived'] = pred_all_rf
sub_rf.to_csv('rf.csv',index=False)

### 4.5 Extra Trees (Extremely Randomized Trees)

Similar to Random Forest, but splits are chosen **fully at random** (no optimal split search), which further reduces variance at the cost of a slight increase in bias. Often faster to train and competitive in accuracy.


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
extr = ExtraTreesClassifier()

param_grid = {'max_depth': [6,7,8,9], 'max_features': [7,8,9,10],
              'n_estimators': [50, 100, 200]}

extr_grid = GridSearchCV(extr, param_grid, cv=10, refit=True, verbose=1)
extr_grid.fit(X_sc,y_sc)
sc_extr = get_best_score(extr_grid)

In [ ]:
plot_feature_importances(extr_grid, X.columns)

In [ ]:
pred_all_extr = extr_grid.predict(X_test_sc)

sub_extr = pd.DataFrame()
sub_extr['PassengerId'] = df_test['PassengerId']
sub_extr['Survived'] = pred_all_extr
sub_extr.to_csv('extr.csv',index=False)

### 4.6 Gradient Boosting Classifier (GBC)

Unlike bagging methods, gradient boosting builds trees **sequentially**, each correcting the residuals of the previous one. This makes it a powerful learner, but more prone to overfitting and slower to train. We tune the number of estimators, tree depth, and minimum samples per split.


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gbc = GradientBoostingClassifier()

param_grid = {'n_estimators': [50, 100],
              'min_samples_split': [3, 4, 5, 6, 7],
              'max_depth': [3, 4, 5, 6]}
gbc_grid = GridSearchCV(gbc, param_grid, cv=10, refit=True, verbose=1)
gbc_grid.fit(X_sc,y_sc)
sc_gbc = get_best_score(gbc_grid)

In [ ]:
plot_feature_importances(gbc_grid, X.columns)

In [ ]:
pred_all_gbc = gbc_grid.predict(X_test_sc)

sub_gbc = pd.DataFrame()
sub_gbc['PassengerId'] = df_test['PassengerId']
sub_gbc['Survived'] = pred_all_gbc
sub_gbc.to_csv('gbc.csv',index=False)

### 4.7 XGBoost

XGBoost is a highly optimized gradient boosting implementation with built-in L1/L2 regularization, missing value handling, and parallel computation. The `gamma` parameter controls tree complexity by requiring a minimum loss reduction to make a split — this acts as a pruning mechanism.


In [ ]:
from xgboost import XGBClassifier
xgb = XGBClassifier()
param_grid = {'max_depth': [5,6,7,8], 'gamma': [1, 2, 4], 'learning_rate': [0.1, 0.2, 0.3, 0.5]}

xgb_grid = GridSearchCV(xgb, param_grid, cv=10, refit=True, verbose=1)
xgb_grid.fit(X_sc,y_sc)
sc_xgb = get_best_score(xgb_grid)

In [ ]:
plot_feature_importances(xgb_grid, X.columns)

In [ ]:
pred_all_xgb = xgb_grid.predict(X_test_sc)

sub_xgb = pd.DataFrame()
sub_xgb['PassengerId'] = df_test['PassengerId']
sub_xgb['Survived'] = pred_all_xgb
sub_xgb.to_csv('xgb.csv',index=False)

### 4.8 AdaBoost

AdaBoost (Adaptive Boosting) builds a strong classifier from a sequence of weak learners (shallow trees). Each subsequent learner focuses more on the misclassified examples of the previous step, by up-weighting them. We tune the number of estimators and learning rate.


In [ ]:
from sklearn.ensemble import AdaBoostClassifier
ada = AdaBoostClassifier()

param_grid = {'n_estimators': [30, 50, 100], 'learning_rate': [0.08, 0.1, 0.2]}
ada_grid = GridSearchCV(ada, param_grid, cv=10, refit=True, verbose=1)
ada_grid.fit(X_sc,y_sc)
sc_ada = get_best_score(ada_grid)

pred_all_ada = ada_grid.predict(X_test_sc)

In [ ]:
plot_feature_importances(ada_grid, X.columns)

In [ ]:
sub_ada = pd.DataFrame()
sub_ada['PassengerId'] = df_test['PassengerId']
sub_ada['Survived'] = pred_all_ada
sub_ada.to_csv('ada.csv',index=False)

### 4.9 CatBoost

CatBoost is a gradient boosting library developed by Yandex with native handling of categorical features and reduced overfitting through ordered boosting. In this dataset all features are already encoded, so we use it as a standard boosting model.


In [ ]:
!pip install catboost
from catboost import CatBoostClassifier
cat=CatBoostClassifier()

param_grid = {'iterations': [100, 150], 'learning_rate': [0.3, 0.4, 0.5], 'loss_function' : ['Logloss']}

cat_grid = GridSearchCV(cat, param_grid, cv=10, refit=True, verbose=1)
cat_grid.fit(X_sc,y_sc, verbose=False)
sc_cat = get_best_score(cat_grid)

pred_all_cat = cat_grid.predict(X_test_sc)

In [ ]:
plot_feature_importances(cat_grid, X.columns)

In [ ]:
sub_cat = pd.DataFrame()
sub_cat['PassengerId'] = df_test['PassengerId']
sub_cat['Survived'] = pred_all_cat
sub_cat['Survived'] = sub_cat['Survived'].astype(int)
sub_cat.to_csv('cat.csv',index=False)

### 4.10 LightGBM

LightGBM uses **leaf-wise** tree growth (vs. level-wise in XGBoost), which can achieve lower loss with fewer iterations. It is particularly efficient on large datasets. We tune `max_depth`, `num_leaves`, `learning_rate`, and `n_estimators`.


In [ ]:
import lightgbm as lgb
lgbm = lgb.LGBMClassifier()
param_grid = {"max_depth": [8,10,15], "learning_rate" : [0.008,0.01,0.012],
              "num_leaves": [80,100,120], "n_estimators": [200,250]  }
lgbm_grid = GridSearchCV(lgbm, param_grid, cv=10, refit=True, verbose=1)
lgbm_grid.fit(X_sc,y_sc)
sc_lgbm = get_best_score(lgbm_grid)

pred_all_lgbm = lgbm_grid.predict(X_test_sc)

In [ ]:
plot_feature_importances(lgbm_grid, X.columns)

In [ ]:
sub_lgbm = pd.DataFrame()
sub_lgbm['PassengerId'] = df_test['PassengerId']
sub_lgbm['Survived'] = pred_all_lgbm
sub_lgbm.to_csv('lgbm.csv',index=False)

### 4.11 Voting Classifier

A Voting Classifier combines the predictions of multiple heterogeneous models. In **soft voting**, the predicted class probabilities are averaged — this generally outperforms hard (majority) voting because it incorporates prediction confidence.

We test two combinations:
- `clf1` (Logistic Regression) + `clf2` (Random Forest) + `clf3` (GaussianNB)
- `clf4` (GBC) + `clf5` (SVC) + `clf6` (Random Forest)


In [ ]:
from sklearn.ensemble import VotingClassifier

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

clf1 = LogisticRegression(random_state=1)
clf2 = RandomForestClassifier(random_state=1)
clf3 = GaussianNB()

eclf = VotingClassifier(estimators=[('lr', clf1), ('rf', clf2), ('gnb', clf3)], voting='soft')

params = {'lr__C': [1.0, 100.0], 'rf__n_estimators': [20, 200],}

votingclf_grid = GridSearchCV(estimator=eclf, param_grid=params, cv=10)
votingclf_grid.fit(X_sc,y_sc)
sc_vot1 = get_best_score(votingclf_grid)

In [ ]:
clf4 = GradientBoostingClassifier()
clf5 = SVC()
clf6 = RandomForestClassifier()

eclf_2 = VotingClassifier(estimators=[('gbdt', clf4),
                                      ('svc', clf5),
                                      ('rf', clf6)], voting='soft')

params = {'gbdt__n_estimators': [50], 'gbdt__min_samples_split': [3],
          'svc__C': [10, 100] , 'svc__gamma': [0.1,0.01,0.001] , 'svc__kernel': ['rbf'] , 'svc__probability': [True],
          'rf__max_depth': [7], 'rf__max_features': [2,3], 'rf__min_samples_split': [3] }

votingclf_grid_2 = GridSearchCV(estimator=eclf_2, param_grid=params, cv=10)
votingclf_grid_2.fit(X_sc,y_sc)
sc_vot2_cv = get_best_score(votingclf_grid_2)

In [ ]:
pred_all_vot2 = votingclf_grid_2.predict(X_test_sc)

sub_vot2 = pd.DataFrame()
sub_vot2['PassengerId'] = df_test['PassengerId']
sub_vot2['Survived'] = pred_all_vot2
sub_vot2.to_csv('vot2.csv',index=False)

### 4.12 Stacking Classifier

Stacking trains a **meta-learner** (here, Logistic Regression) on top of the out-of-fold predictions of multiple base classifiers. Unlike voting, the meta-learner learns *how to weight* each base model's contribution based on the training data — in theory providing a more adaptive combination.

Base models: XGBoost, GBC, Random Forest, SVC.


In [ ]:
from mlxtend.classifier import StackingClassifier

In [ ]:
# Initializing models
clf1 = xgb_grid.best_estimator_
clf2 = gbc_grid.best_estimator_
clf3 = rf_grid.best_estimator_
clf4 = svc_grid.best_estimator_

lr = LogisticRegression()
st_clf = StackingClassifier(classifiers=[clf1, clf1, clf2, clf3, clf4], meta_classifier=lr)

params = {'meta_classifier__C': [0.1,1.0,5.0,10.0] ,
          #'use_probas': [True] ,
          #'average_probas': [True] ,
          'use_features_in_secondary' : [True, False]
         }
st_clf_grid = GridSearchCV(estimator=st_clf, param_grid=params, cv=5, refit=True)
st_clf_grid.fit(X_sc, y_sc)
sc_st_clf = get_best_score(st_clf_grid)

In [ ]:
pred_all_stack = st_clf_grid.predict(X_test_sc)

sub_stack = pd.DataFrame()
sub_stack['PassengerId'] = df_test['PassengerId']
sub_stack['Survived'] = pred_all_stack
sub_stack.to_csv('stack_clf.csv',index=False)

---
## 5. Model Comparison and Selection

We compare all trained models on two axes:
1. **Cross-validation accuracy** on the training set (10-fold)
2. **Kaggle leaderboard score** (submission accuracy on the private test set)

A large gap between CV score and submission score indicates overfitting. The SVC achieved the best Kaggle score (`0.8086`), despite not having the highest CV score — a reminder that cross-validation is an estimate, not a guarantee.


In [ ]:
list_scores = [sc_knn, sc_rf, sc_extr, sc_svc, sc_gbc, sc_xgb,
               sc_ada, sc_cat, sc_lgbm, sc_vot2_cv, sc_st_clf]

list_classifiers = ['KNN','RF','EXTR','SVC','GBC','XGB',
                    'ADA','CAT','LGBM','VOT2','STACK']

In [ ]:
score_subm_svc   = 0.80861
score_subm_vot2  = 0.78947
score_subm_ada   = 0.78468
score_subm_lgbm  = 0.78468
score_subm_rf    = 0.77990
score_subm_xgb   = 0.77033
score_subm_dtree = 0.76076
score_subm_extr  = 0.76076
score_subm_gbc  = 0.74641
score_subm_cat   = 0.74162
score_subm_knn   = 0.69856

score_subm_stack = 0.76076

In [ ]:
subm_scores = [score_subm_knn, score_subm_rf, score_subm_extr, score_subm_svc,
               score_subm_gbc, score_subm_xgb, score_subm_ada, score_subm_cat,
               score_subm_lgbm, score_subm_vot2, score_subm_stack]

In [ ]:
trace1 = go.Scatter(x=list_classifiers, y=list_scores,
                    name="Validation", text=list_classifiers,
                    mode='lines+markers', marker=dict(size=8))
trace2 = go.Scatter(x=list_classifiers, y=subm_scores,
                    name="Submission", text=list_classifiers,
                    mode='lines+markers', marker=dict(size=8))

fig = go.Figure(data=[trace1, trace2])

fig.update_layout(
    title=dict(text="Validation and Submission Scores", x=0.5),
    xaxis=dict(ticklen=10, zeroline=False),
    yaxis=dict(title="Accuracy", ticklen=10),
    legend=dict(orientation="v", x=1.05, y=1.0),
    width=750,
    height=500,
)

fig.show(renderer="colab")

### 5.2 Prediction Correlation Between Models

We compute the correlation matrix of the predictions made by each model on the test set. High correlation between models means they tend to agree (and disagree) on the same passengers — reducing the potential benefit of ensembling them. For ensembles to work well, we want base models that are **diverse** (low correlation).


In [ ]:
predictions = {'KNN': pred_all_knn, 'RF': pred_all_rf, 'EXTR': pred_all_extr,
               'SVC': pred_all_svc, 'GBC': pred_all_gbc, 'XGB': pred_all_xgb,
               'ADA': pred_all_ada, 'CAT': pred_all_cat, 'LGBM': pred_all_lgbm,
               'VOT2': pred_all_vot2, 'STACK': pred_all_stack}
df_predictions = pd.DataFrame(data=predictions)
df_predictions.corr()

In [ ]:
plt.figure(figsize=(9, 9))
sns.set(font_scale=1.25)
sns.heatmap(df_predictions.corr(), linewidths=1.5, annot=True, square=True,
                fmt='.2f', annot_kws={'size': 10},
                yticklabels=df_predictions.columns , xticklabels=df_predictions.columns
            )
plt.yticks(rotation=0)
plt.show()

### 5.3 Feature Importance Comparison Across Ensemble Models

We visualize and compare feature importances across all tree-based models. Consistent high importance across models gives us confidence that a feature is genuinely predictive. Variables that appear important in only one model may be artifacts of overfitting.

> **Key finding:** `Title`, `Fare`, `Age_bin`, and `FamilySize` consistently rank as the most informative features across models.


In [ ]:
imp_rf   = pd.Series(data = rf_grid.best_estimator_.feature_importances_, index=X.columns)
imp_extr = pd.Series(data = extr_grid.best_estimator_.feature_importances_, index=X.columns)
imp_gbc = pd.Series(data = gbc_grid.best_estimator_.feature_importances_, index=X.columns)
imp_xgb = pd.Series(data = xgb_grid.best_estimator_.feature_importances_, index=X.columns)
imp_ada = pd.Series(data = ada_grid.best_estimator_.feature_importances_, index=X.columns)
imp_cat = pd.Series(data = cat_grid.best_estimator_.feature_importances_ / 100, index=X.columns)
imp_lgbm = pd.Series(data = lgbm_grid.best_estimator_.feature_importances_ / 10000, index=X.columns)
imp_lgbm /= imp_lgbm.sum()

In [ ]:
models=['RF', 'EXTR', 'GBC', 'XGB', 'ADA', 'CAT', 'LGBM']

fig = go.Figure(data=[
    go.Bar(name=models[0],  x=imp_rf.index,   y=imp_rf.values),
    go.Bar(name=models[1],  x=imp_extr.index, y=imp_extr.values),
    go.Bar(name=models[2],  x=imp_gbc.index,  y=imp_gbc.values),
    go.Bar(name=models[3],  x=imp_xgb.index,  y=imp_xgb.values),
    go.Bar(name=models[4],  x=imp_ada.index,  y=imp_ada.values),
    go.Bar(name=models[5],  x=imp_cat.index,  y=imp_cat.values),
    go.Bar(name=models[6],  x=imp_lgbm.index, y=imp_lgbm.values),
])

fig.update_layout(
    barmode='group',
    title=go.layout.Title(text="Feature Importances for Ensemble Models", xref="paper", x=0.5),
    xaxis_title="Feature",
    yaxis_title="Importance",
    legend_title="Model",
    height=500,
)

fig.show(renderer="colab")

---
## 5.4 Confusion Matrix & Classification Report — Best Model

Cross-validation accuracy tells us the average performance, but it hides **what kinds of errors** the model makes. A confusion matrix breaks down predictions into:

- **True Positives (TP):** Correctly predicted survivors
- **True Negatives (TN):** Correctly predicted non-survivors  
- **False Positives (FP):** Predicted survived, actually died *(Type I error)*
- **False Negatives (FN):** Predicted died, actually survived *(Type II error)*

From these we derive three complementary metrics beyond raw accuracy:

| Metric | Formula | Meaning |
|---|---|---|
| **Precision** | TP / (TP + FP) | Of all predicted survivors, how many actually survived? |
| **Recall** | TP / (TP + FN) | Of all actual survivors, how many did we catch? |
| **F1 Score** | 2 × (P × R) / (P + R) | Harmonic mean of precision and recall |

We evaluate the **best-performing model** (SVC) using stratified cross-validation to produce a reliable confusion matrix.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, roc_curve, auc,
                             precision_recall_curve)

# Use the best model: SVC with tuned parameters
best_model = svc_grid.best_estimator_

# Out-of-fold predictions via stratified 10-fold CV
y_pred_cv = cross_val_predict(best_model, X_sc, y_sc, cv=10)
y_prob_cv = cross_val_predict(best_model, X_sc, y_sc, cv=10, method='decision_function')

# ── Confusion Matrix ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_sc, y_pred_cv)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Did not survive', 'Survived'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — SVC (10-fold CV)', fontsize=13, fontweight='bold', pad=10)

# Annotate with rates
tn, fp, fn, tp = cm.ravel()
total = cm.sum()
axes[0].text(0, 0, f'TN\n{tn} ({tn/total:.1%})', ha='center', va='center',
             fontsize=11, color='white', fontweight='bold')
axes[0].text(1, 0, f'FP\n{fp} ({fp/total:.1%})', ha='center', va='center',
             fontsize=11, color='black', fontweight='bold')
axes[0].text(0, 1, f'FN\n{fn} ({fn/total:.1%})', ha='center', va='center',
             fontsize=11, color='black', fontweight='bold')
axes[0].text(1, 1, f'TP\n{tp} ({tp/total:.1%})', ha='center', va='center',
             fontsize=11, color='white', fontweight='bold')

# ── ROC Curve ────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_sc, y_prob_cv)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#2196F3', lw=2, label=f'SVC (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
axes[1].fill_between(fpr, tpr, alpha=0.08, color='#2196F3')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curve — SVC', fontsize=13, fontweight='bold', pad=10)
axes[1].legend(loc='lower right', fontsize=11)
axes[1].set_xlim([-0.01, 1.01])
axes[1].set_ylim([-0.01, 1.01])
sns.despine(ax=axes[1])

plt.tight_layout()
plt.show()

# ── Classification Report ────────────────────────────────────────────────────
print("\n" + "="*55)
print("       CLASSIFICATION REPORT — SVC (10-fold CV)")
print("="*55)
print(classification_report(y_sc, y_pred_cv, target_names=['Did not survive', 'Survived']))
print(f"ROC-AUC Score: {roc_auc:.4f}")


### 5.5 Full Metrics Comparison Across All Models

Accuracy alone is an incomplete picture. We evaluate all models on **Precision, Recall, F1, and ROC-AUC** using cross-validation, allowing a more nuanced comparison — especially important since the Titanic dataset has a moderate class imbalance (~38% survivors vs. ~62% non-survivors).


In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, roc_curve, auc)

best_model = svc_grid.best_estimator_

y_pred_cv = cross_val_predict(best_model, X_sc, y_sc, cv=10)
y_prob_cv = cross_val_predict(best_model, X_sc, y_sc, cv=10, method='decision_function')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_sc, y_pred_cv)
tn, fp, fn, tp = cm.ravel()
total = cm.sum()

# values_format='' desactiva las anotaciones automáticas de ConfusionMatrixDisplay
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Did not survive', 'Survived'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues', values_format='')
axes[0].set_title('Confusion Matrix — SVC (10-fold CV)', fontsize=13, fontweight='bold', pad=10)

annotations = [
    (0, 0, f'TN\n{tn}\n({tn/total:.1%})', 'white'),
    (1, 0, f'FP\n{fp}\n({fp/total:.1%})', 'black'),
    (0, 1, f'FN\n{fn}\n({fn/total:.1%})', 'black'),
    (1, 1, f'TP\n{tp}\n({tp/total:.1%})', 'white'),
]
for x, y_pos, text, color in annotations:
    axes[0].text(x, y_pos, text, ha='center', va='center',
                 fontsize=11, color=color, fontweight='bold', linespacing=1.5)

fpr, tpr, _ = roc_curve(y_sc, y_prob_cv)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#2196F3', lw=2, label=f'SVC (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
axes[1].fill_between(fpr, tpr, alpha=0.08, color='#2196F3')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curve — SVC', fontsize=13, fontweight='bold', pad=10)
axes[1].legend(loc='lower right', fontsize=11)
axes[1].set_xlim([-0.01, 1.01])
axes[1].set_ylim([-0.01, 1.01])
sns.despine(ax=axes[1])

plt.tight_layout()
plt.show()

print("\n" + "="*55)
print("       CLASSIFICATION REPORT — SVC (10-fold CV)")
print("="*55)
print(classification_report(y_sc, y_pred_cv, target_names=['Did not survive', 'Survived']))
print(f"ROC-AUC Score: {roc_auc:.4f}")


---
## 6. Model Interpretability — SHAP Values

Standard feature importances from tree-based models (e.g. Gini impurity reduction) answer the question *"which features does the model use most?"* — but they don't tell us **in which direction** or **for which specific passengers**.

[SHAP (SHapley Additive exPlanations)](https://shap.readthedocs.io/) is a game-theoretic framework that assigns each feature a contribution to the model's prediction for **each individual sample**. This gives us both global and local interpretability:

| Analysis | Question answered |
|---|---|
| **Beeswarm plot** | Which features matter most globally, and in which direction? |
| **Waterfall plot** | Why did the model predict survival/death for *this specific passenger*? |
| **Dependence plot** | How does the model's output change as a feature increases? |
| **Force plot** | Visual breakdown of a single prediction |

We use XGBoost as the base model since `shap.TreeExplainer` is optimized for tree-based models and produces exact (not approximate) SHAP values.


In [ ]:
# Install SHAP if not already available
!pip install shap -q


In [ ]:
import shap
shap.initjs()

# ── Refit XGBoost sobre datos NO escalados ───────────────────────────────────
# SHAP necesita los valores originales de las features para colorear y graficar
# correctamente. Los scaled values producen colores grises y ejes vacíos.
xgb_shap = xgb_grid.best_estimator_
xgb_shap.fit(X, y)  # <-- X sin escalar, no X_sc

explainer = shap.TreeExplainer(xgb_shap)

# DataFrame con nombres de columnas (imprescindible para waterfall y dependence)
feature_names = X.columns.tolist()

# Convert boolean columns to int to avoid 'object' dtype issues with XGBoost/SHAP
X_for_shap = X.copy()
for col in X_for_shap.select_dtypes(include='bool').columns:
    X_for_shap[col] = X_for_shap[col].astype(int)

# Now X_df will have numerical dtypes as expected
X_df = X_for_shap

shap_values = explainer.shap_values(X_df)
print(f"SHAP values shape: {shap_values.shape}")
print(f"Features: {feature_names}")

### 6.1 Global Feature Importance — Beeswarm Plot

The beeswarm plot shows **all SHAP values for all samples** simultaneously:
- Each dot represents one passenger
- **Position on x-axis**: magnitude and direction of the feature's contribution to the prediction
  - Positive SHAP → pushed prediction toward *survived*
  - Negative SHAP → pushed prediction toward *did not survive*
- **Color**: the feature's actual value (red = high, blue = low)

This is far more informative than a simple bar chart of importances — it reveals both *how much* and *in which direction* each feature influences the model.


In [ ]:
# ── 6.1 Beeswarm ─────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_df, plot_type="dot",
                  show=False, max_display=15)
plt.title("SHAP Beeswarm Plot — XGBoost\nImpact of Each Feature on Model Output",
          fontsize=13, fontweight='bold', pad=14)
plt.tight_layout()
plt.show()


### 6.2 Mean Absolute SHAP — Global Feature Ranking

The bar chart shows the **mean absolute SHAP value** per feature — a clean, aggregated view of global importance that averages out direction and is directly comparable across features.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
shap.summary_plot(shap_values, X_sc_df, plot_type="bar",
                  show=False, max_display=15)
plt.title("Mean |SHAP Value| per Feature — XGBoost",
          fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


### 6.3 Dependence Plots — Feature Interaction Effects

A dependence plot shows how the SHAP value for one feature changes as its value increases, colored by a second interacting feature. This reveals **non-linear relationships and interaction effects** that neither a simple correlation nor a bar chart can show.

We plot the three most important features: `Title`, `Fare`, and `Age_bin`.


In [ ]:
# ── 6.3 Dependence plots ─────────────────────────────────────────────────────
top_features = ['Title', 'Fare', 'Age_bin']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feat in enumerate(top_features):
    if feat not in feature_names:
        feat = feature_names[i]
    shap.dependence_plot(feat, shap_values, X_df,
                         ax=axes[i], show=False, dot_size=20)
    axes[i].set_title(f'SHAP Dependence — {feat}',
                      fontsize=12, fontweight='bold')
    sns.despine(ax=axes[i])

plt.suptitle('SHAP Dependence Plots — Feature Effects & Interactions',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### 6.4 Local Interpretability — Waterfall Plots

While the previous plots explain the model **globally**, waterfall plots explain **individual predictions**: why did the model predict that *this specific passenger* survived or did not?

Each bar shows how much a feature pushed the prediction up (red) or down (blue) from the **base value** (the average model output across all training samples) to the final prediction.

We examine 4 passengers: 2 survivors and 2 non-survivors, to understand what drives each prediction.


In [ ]:
# ── 6.4 Waterfall plots ──────────────────────────────────────────────────────
survived_idx     = y[y == 1].index[:2].tolist()
not_survived_idx = y[y == 0].index[:2].tolist()
sample_indices   = survived_idx + not_survived_idx
labels = ['Survivor #1', 'Survivor #2', 'Non-survivor #1', 'Non-survivor #2']

# Construir Explanation con los índices posicionales, no los de pandas
X_arr = X_df.values
pos_map = {idx: pos for pos, idx in enumerate(X_df.index)}

explanation = shap.Explanation(
    values      = shap_values,
    base_values = explainer.expected_value,
    data        = X_arr,
    feature_names = feature_names
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for ax, idx, label in zip(axes, sample_indices, labels):
    pos = pos_map[idx]  # posición posicional en el array
    plt.sca(ax)
    shap.waterfall_plot(explanation[pos], max_display=10, show=False)
    actual = "Survived" if y[idx] == 1 else "Did not survive"
    ax.set_title(f'{label} (Passenger {idx}) — Actual: {actual}',
                 fontsize=11, fontweight='bold', pad=8)

plt.suptitle('SHAP Waterfall Plots — Individual Prediction Explanations',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# En Colab, para hacer push:
!git add Titanic_Practice.ipynb
!git commit -m "Update notebook"
!git push origin main

### 6.5 Key Takeaways from SHAP Analysis

Based on the SHAP analysis, we can draw the following **model-backed conclusions**:

1. **`Title`** is the single most impactful feature — titles like *Mrs* and *Miss* strongly push predictions toward survival, while *Mr* strongly pushes toward non-survival. This encodes both gender and social status simultaneously.

2. **`Fare`** has a strong positive relationship with survival — higher-paying passengers (1st class) had significantly better odds, as captured by the dependence plot's upward trend.

3. **`Age_bin`** shows a non-linear effect — very young passengers (children) received a survival boost, while the elderly showed mixed results depending on class.

4. **`Sex_male`** (encoded as 1 for male) consistently produces large negative SHAP values — confirming the "women and children first" protocol quantitatively.

5. **`Pclass`** and `FamilySize` contribute moderate but consistent effects — 1st class and medium-sized families had survival advantages.

> **Why SHAP over feature importances?** Standard `feature_importances_` only tells us average usage across all splits. SHAP tells us the *direction* and *magnitude* of each feature's effect for *every individual passenger* — making the model auditable and its decisions explainable.
